# [LAB-08] 4. 다변량 분석

## #01. 준비작업

### 1. 이전 분석 내용 가져오기

In [1]:
%%capture cap
%run "./[LAB-08] PBT - EDAㅣ1-EDA 시작하기(실습코드).ipynb"

### 2. 불러온 내용 확인

- 이전 분석 내역이 잘 로드 되었는지 확인한다.

In [2]:
print("종속변수 :", target)
print("종속변수 유형: " + ("연속형" if target_is_continuous else "명목형"))
print("연속형   :", continuous_cols)
print("명목형   :", nominal_cols)

display(df.head())
display(desc.head())
display(cat_desc.head())

종속변수 : MEDV
종속변수 유형: 연속형
연속형   : ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
명목형   : ['CHAS']


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.006,18.000,2.310,0,0.538,6.575,65.200,4.090,1,296,15.300,396.900,4.980,24.000
1,0.027,0.000,7.070,0,0.469,6.421,78.900,4.967,2,242,17.800,396.900,9.140,21.600
2,0.027,0.000,7.070,0,0.469,7.185,61.100,4.967,2,242,17.800,392.830,4.030,34.700
3,0.032,0.000,2.180,0,0.458,6.998,45.800,6.062,3,222,18.700,394.630,2.940,33.400
4,0.069,0.000,2.180,0,0.458,7.147,54.200,6.062,3,222,18.700,396.900,5.330,36.200


,count,mean,std,min,25%,50%,75%,max,rel_diff,rdiff_flag,iqr,upper_bound,lower_bound,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio,skew,skew_interpret,kurt,kurt_interpret,log_need
CRIM,506.000,3.614,8.602,0.006,0.082,0.257,3.677,88.976,13.087,large_diff,3.595,9.070,-5.311,66,0.130,0,0.000,66,0.130,5.223,right tail,37.131,leptokurtic,log1p
ZN,506.000,11.364,23.322,0.000,0.000,0.000,12.500,100.000,inf,large_diff,12.500,31.250,-18.750,68,0.134,0,0.000,68,0.134,2.226,right tail,4.032,leptokurtic,log1p
INDUS,506.000,11.137,6.860,0.460,5.190,9.690,18.100,27.740,0.149,diff,12.910,37.465,-14.175,0,0.000,0,0.000,0,0.000,0.295,symmetric,-1.234,platykurtic,none
NOX,506.000,0.555,0.116,0.385,0.449,0.538,0.624,0.871,0.031,similar,0.175,0.886,0.187,0,0.000,0,0.000,0,0.000,0.729,right tail,-0.065,platykurtic,none
RM,506.000,6.285,0.703,3.561,5.885,6.208,6.623,8.780,0.012,similar,0.738,7.731,4.778,22,0.043,8,0.016,30,0.059,0.404,symmetric,1.892,leptokurtic,none


,CHAS
count,506
unique,2
top,0
freq,471


### 3. 라이브러리 참조

In [3]:
from IPython.display import display, Markdown
from pandas import concat, merge
import numpy as np

## #02. 연속형 변수간의 다변량 분석

### 1. 독립변수간 상관 분석

- 각 독립변수와 종속변수간의 상관 분석은 이미 진행된 상태이므로 종속변수를 포함하지 않는다.
  - 다중공선성 신호를 감지하기 위해 사용한다.
- 모델링용 전처리 과정에서 VIF값을 통한 필터링이 진행되므로 사실상 생략해도 무관하다.
- 시각화 결과물이 복잡하고 해석이 어렵기 때문에 PairPlot은 잘 사용하지 않는다.
  - `plot=False` 파라미터 적용 (여기서는 경험을 위해 사용함)

In [4]:
# 연속형 독립변수간의 상관분석
# -> 상관행렬 히트맵과 PairPlot을 출력하고 상관분석 결과를 리턴한다.
corr = my_stats.multi_correlation(origin,columns=continuous_cols, 
                    plot=True, reg=True, width=4800, height=4800)

display(corr)   # 상관분석 결과 출력

,CRIM,ZN,INDUS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
CRIM,1.000,-0.572,0.736,0.822,-0.309,0.704,-0.745,0.728,0.729,0.465,-0.361,0.635
ZN,-0.572,1.000,-0.643,-0.635,0.361,-0.544,0.615,-0.279,-0.371,-0.449,0.163,-0.490
INDUS,0.736,-0.643,1.000,0.791,-0.415,0.679,-0.757,0.456,0.664,0.434,-0.286,0.639
NOX,0.822,-0.635,0.791,1.000,-0.310,0.795,-0.880,0.586,0.649,0.391,-0.297,0.637
RM,-0.309,0.361,-0.415,-0.310,1.000,-0.278,0.263,-0.107,-0.272,-0.313,0.054,-0.641
AGE,0.704,-0.544,0.679,0.795,-0.278,1.000,-0.802,0.418,0.526,0.355,-0.228,0.657
DIS,-0.745,0.615,-0.757,-0.880,0.263,-0.802,1.000,-0.496,-0.574,-0.322,0.250,-0.564
RAD,0.728,-0.279,0.456,0.586,-0.107,0.418,-0.496,1.000,0.705,0.318,-0.282,0.394
TAX,0.729,-0.371,0.664,0.649,-0.272,0.526,-0.574,0.705,1.000,0.453,-0.330,0.534
PTRATIO,0.465,-0.449,0.434,0.391,-0.313,0.355,-0.322,0.318,0.453,1.000,-0.072,0.467


method   coef  p-value  strength  significant  normality_x  \
x       y                                                                       
CRIM    ZN       Spearman -0.572    0.000  Moderate         True        False   
        INDUS    Spearman  0.736    0.000    Strong         True        False   
        NOX      Spearman  0.822    0.000    Strong         True        False   
        RM       Spearman -0.309    0.000  Moderate         True        False   
        AGE      Spearman  0.704    0.000    Strong         True        False   
        DIS      Spearman -0.745    0.000    Strong         True        False   
        RAD      Spearman  0.728    0.000    Strong         True        False   
        TAX      Spearman  0.729    0.000    Strong         True        False   
        PTRATIO  Spearman  0.465    0.000  Moderate         True        False   
        B        Spearman -0.361    0.000  Moderate         True        False   
        LSTAT    Spearman  0.635    0.000  Moderate         True        False   
ZN      INDUS    Spearman -0.643    0.000  Moderate         True        False   
        NOX      Spearman -0.635    0.000  Moderate         True        False   
        RM       Spearman  0.361    0.000  Moderate         True        False   
        AGE      Spearman -0.544    0.000  Moderate         True        False   
        DIS      Spearman  0.615    0.000  Moderate         True        False   
        RAD      Spearman -0.279    0.000      Weak         True        False   
        TAX      Spearman -0.371    0.000  Moderate         True        False   
        PTRATIO  Spearman -0.449    0.000  Moderate         True        False   
        B        Spearman  0.163    0.000      Weak         True        False   
        LSTAT    Spearman -0.490    0.000  Moderate         True        False   
INDUS   NOX      Spearman  0.791    0.000    Strong         True        False   
        RM       Spearman -0.415    0.000  Moderate         True        False   
        AGE      Spearman  0.679    0.000  Moderate         True        False   
        DIS      Spearman -0.757    0.000    Strong         True        False   
        RAD      Spearman  0.456    0.000  Moderate         True        False   
        TAX      Spearman  0.664    0.000  Moderate         True        False   
        PTRATIO  Spearman  0.434    0.000  Moderate         True        False   
        B        Spearman -0.286    0.000      Weak         True        False   
        LSTAT    Spearman  0.639    0.000  Moderate         True        False   
NOX     RM       Spearman -0.310    0.000  Moderate         True        False   
        AGE      Spearman  0.795    0.000    Strong         True        False   
        DIS      Spearman -0.880    0.000    Strong         True        False   
        RAD      Spearman  0.586    0.000  Moderate         True        False   
        TAX      Spearman  0.649    0.000  Moderate         True        False   
        PTRATIO  Spearman  0.391    0.000  Moderate         True        False   
        B        Spearman -0.297    0.000      Weak         True        False   
        LSTAT    Spearman  0.637    0.000  Moderate         True        False   
RM      AGE      Spearman -0.278    0.000      Weak         True        False   
        DIS      Spearman  0.263    0.000      Weak         True        False   
        RAD      Spearman -0.107    0.016      Weak         True        False   
        TAX      Spearman -0.272    0.000      Weak         True        False   
        PTRATIO  Spearman -0.313    0.000  Moderate         True        False   
        B        Spearman  0.054    0.228      Weak        False        False   
        LSTAT    Spearman -0.641    0.000  Moderate         True        False   
AGE     DIS      Spearman -0.802    0.000    Strong         True        False   
        RAD      Spearman  0.418    0.000  Moderate         True        False   
        TAX      Spearman  0.526    0.000  Moderat

## #03. 상관분석 결과표 정리 

### 1. 결과표를 양방향으로 만들기

- corr는 각 쌍이 한 번만 등장하므로(CRIM-NOX만 있고 NOX-CRIM은 없음) x/y를 뒤집어 붙여 양방향으로 만든다

In [5]:
pairs = corr.reset_index()

# 인덱스를 해제한 원본 상관분석 결과표와 x, y를 바꾼 상관분석 결과표를 결합
pairs = concat([pairs, pairs.rename(columns={'x': 'y', 'y': 'x'})])

# 상관계수의 절대값이 큰 순으로 정렬
pairs.sort_values(by='coef', key=abs, ascending=False, inplace=True)

pairs.head(10)

,x,y,method,coef,p-value,strength,significant,normality_x,normality_y,linearity,influential_outlier,high_skew
32,DIS,NOX,Spearman,-0.880,0.000,Strong,True,False,False,False,False,True
32,NOX,DIS,Spearman,-0.880,0.000,Strong,True,False,False,False,False,True
2,NOX,CRIM,Spearman,0.822,0.000,Strong,True,False,False,False,True,True
2,CRIM,NOX,Spearman,0.822,0.000,Strong,True,False,False,False,True,True
45,DIS,AGE,Spearman,-0.802,0.000,Strong,True,False,False,False,False,True
45,AGE,DIS,Spearman,-0.802,0.000,Strong,True,False,False,False,False,True
31,NOX,AGE,Spearman,0.795,0.000,Strong,True,False,False,False,False,False
31,AGE,NOX,Spearman,0.795,0.000,Strong,True,False,False,False,False,False
21,INDUS,NOX,Spearman,0.791,0.000,Strong,True,False,False,False,False,False
21,NOX,INDUS,Spearman,0.791,0.000,Strong,True,False,False,False,False,False


### 2. 변수별 집계표

- 변수별로 가장 상관계수가 큰 순서대로 정렬했으므로    
  그룹별 첫 번째 항목을 가져오면 각 변수별로 가장 상관계수가 큰 변수를 확인할 수 있다.

In [6]:
# 앞 단계에서 상관계수에 대해 내림차순 정렬되어 있는 상태로
# x별로 그룹화 하여 가장 첫 번째 항목 선택 --> 가장 상관계수가 큰 항목
summary = pairs.groupby('x').first()

# 컬럼 이름 수정
summary.rename(columns={'y': 'max-y', 'coef': 'max-coef'}, inplace=True)
summary

,max-y,method,max-coef,p-value,strength,significant,normality_x,normality_y,linearity,influential_outlier,high_skew
x,,,,,,,,,,,
AGE,DIS,Spearman,-0.802,0.000,Strong,True,False,False,False,False,True
B,CRIM,Spearman,-0.361,0.000,Moderate,True,False,False,False,True,True
CRIM,NOX,Spearman,0.822,0.000,Strong,True,False,False,False,True,True
DIS,NOX,Spearman,-0.880,0.000,Strong,True,False,False,False,False,True
INDUS,NOX,Spearman,0.791,0.000,Strong,True,False,False,False,False,False
LSTAT,AGE,Spearman,0.657,0.000,Moderate,True,False,False,False,False,False
NOX,DIS,Spearman,-0.880,0.000,Strong,True,False,False,False,False,True
PTRATIO,LSTAT,Spearman,0.467,0.000,Moderate,True,False,False,False,False,False
RAD,CRIM,Spearman,0.728,0.000,Strong,True,False,False,False,True,True


### 3. 상관정도가 강한 쌍에 대한 집계

- 양방향 결과표에서 x별로 그룹을 묶고, x가 동일한 y에 대해서 갯수와 쉼표로 연결한 y이름을 구한다.

In [7]:
# 양방향 결과표에서 상관정도가 강한 항목만 추출
strong = pairs[pairs['strength'] == 'Strong']

# 추출된 결과에서 x별로 그룹화하여 y의 갯수와 y를 콤마로 연결한 문자열 집계
group_pairs = strong.groupby('x')['y'].agg(['count', ', '.join])
group_pairs

,count,join
x,,
AGE,3,"DIS, NOX, CRIM"
CRIM,6,"NOX, DIS, INDUS, TAX, RAD, AGE"
DIS,4,"NOX, AGE, INDUS, CRIM"
INDUS,3,"NOX, DIS, CRIM"
NOX,4,"DIS, CRIM, AGE, INDUS"
RAD,2,"CRIM, TAX"
TAX,2,"CRIM, RAD"


### 4. 상관분석 결과표 정리

- 변수별 집계표(summary)와 상관정도가 강한 쌍에 대한 집계(group_pairs)를 병합한다.

In [8]:
# 집계표 x에 대한 상관정도가 가장 큰 y 이름과 상관계수 추출
s = summary[['max-y', 'max-coef']]

# "추출된 표"를 기준으로 "상관 정도가 강한 쌍에 대한 집계표" 병합
corr_table = merge(s, group_pairs, left_index=True, right_index=True, how='left')

# 상관정도가 강한 변수가 없을 경우 갯수는 0, y는 "-"로 표시
corr_table.fillna({'count': 0, 'join': "-"}, inplace=True)

# 상관정도가 강한 변수의 갯수와 상관계수의 절대값이 큰 순으로 정렬
corr_table.sort_values(by=['count', 'max-coef'], key=abs, ascending=False, inplace=True)

# 컬럼 이름 수정
corr_table.rename(columns={'join': 'columns'}, inplace=True)

corr_table

,max-y,max-coef,count,columns
x,,,,
CRIM,NOX,0.822,6.000,"NOX, DIS, INDUS, TAX, RAD, AGE"
DIS,NOX,-0.880,4.000,"NOX, AGE, INDUS, CRIM"
NOX,DIS,-0.880,4.000,"DIS, CRIM, AGE, INDUS"
AGE,DIS,-0.802,3.000,"DIS, NOX, CRIM"
INDUS,NOX,0.791,3.000,"NOX, DIS, CRIM"
TAX,CRIM,0.729,2.000,"CRIM, RAD"
RAD,CRIM,0.728,2.000,"CRIM, TAX"
LSTAT,AGE,0.657,0.000,-
ZN,INDUS,-0.643,0.000,-


#### 💡 인사이트

> 이변량 분석의 후보는 **각 변수와 `MEDV`의 단독 관계**만 보고 뽑은 것이다.
> **이 단계가 변수를 제외하는 단계다.** 위 결과표의 `count`(얽힌 상대의 수)와 `columns`(얽힌 상대의 이름)를 읽어 판단한다.

**제외 기준** — 아래 **두 조건을 모두** 만족하면 제외 후보가 된다

1. `count` ≥ 1 → 다른 독립변수와 \|ρ\|≥0.7로 얽혀 있다 (**같은 정보의 중복**)
2. 얽힌 상대들보다 **종속변수(`MEDV`)와의 관계가 약하다** (**대체 가능**)

> 두 조건이 함께 필요하다. `count`만 크면 "많이 얽혔다"일 뿐이고, 얽힌 무리 안에서 **가장 강한 하나는 대표로 남겨야** 정보가 사라지지 않는다.
> `count`=0이면 대체할 상대 자체가 없으므로 **제외 대상이 아니다.**

**1) `count`로 세 무리가 갈린다**

| `count` | 변수 | 읽는 법 |
|---|---|---|
| 6 ~ 3 | `CRIM`·`DIS`·`NOX`·`AGE`·`INDUS` | 다섯 변수의 `columns`가 **서로를 지목**한다 → 사실상 한 덩어리 |
| 2 | `TAX`·`RAD` | 둘의 `columns`가 **서로를 지목**한다 (`CRIM, RAD` / `CRIM, TAX`) → 한 덩어리 |
| 0 | `LSTAT`·`ZN`·`RM`·`PTRATIO`·`B` | `columns`가 `-` → **얽힌 상대 없음 → 기준 미해당** |

**2) 덩어리마다 대표 하나만 남긴다** — 기준은 이변량에서 구한 `MEDV`와의 효과크기

| 덩어리 | 남길 변수 | 근거 |
|---|---|---|
| `CRIM`·`DIS`·`NOX`·`AGE`·`INDUS` | **`INDUS`** | 다섯 중 `MEDV`와 가장 강함 (ρ=-0.578) |
| `TAX`·`RAD` | **`TAX`** | ρ=-0.562 > `RAD` ρ=-0.347 |
| `size`=0인 다섯 개 | **전부 유지** | 얽힌 상대가 없어 서로 다른 정보를 담고 있다 |

**→ 두 조건을 모두 만족하여 제외 대상이 되는 변수: `CRIM`·`DIS`·`NOX`·`AGE`·`RAD` (5개)**

**주의**

- 여기서 본 것은 **2변량 신호뿐**이다. 3개 이상이 결합해 생기는 공선성은 잡히지 않으므로 **최종 판정은 VIF 단계로 넘긴다.**


## #04. 상관분석 결과표 정리 모듈화 기능 확인

### 1. 다중공선성 신호 감지

In [9]:
my_stats.correlation_summary(corr)

,max-y,max-coef,count,columns
x,,,,
CRIM,NOX,0.822,6,"NOX, DIS, INDUS, TAX, RAD, AGE"
DIS,NOX,-0.880,4,"NOX, AGE, INDUS, CRIM"
NOX,DIS,-0.880,4,"DIS, CRIM, AGE, INDUS"
AGE,DIS,-0.802,3,"DIS, NOX, CRIM"
INDUS,NOX,0.791,3,"NOX, DIS, CRIM"
TAX,CRIM,0.729,2,"CRIM, RAD"
RAD,CRIM,0.728,2,"CRIM, TAX"
LSTAT,AGE,0.657,0,-
ZN,INDUS,-0.643,0,-


## 최종 변수 선택

- 이변량 분석에서 채택된 후보에 **다변량 분석(독립변수 간 중복)** 결과를 반영하여 갱신한다.
- **갱신 규칙**: 강한 쌍(\|ρ\|≥0.7)으로 묶인 변수끼리는 **종속변수와의 효과크기가 가장 큰 대표 1개만 남기고** 나머지는 내린다.

| 채택여부 | 변수 | 효과크기 | 중복 | 근거 |
|---|---|---|---|---|
| ✅ 채택 | LSTAT | ρ=-0.853 (강함) | 없음 | 종속변수와 최강 + 중복 없음 → **1순위** |
| ✅ 채택 | RM | ρ=+0.634 (중간) | 없음 | 유의 + 중복 없음 → 독립적 정보 |
| ✅ 채택 | INDUS | ρ=-0.578 (중간) | 도심축 | **도심축 5개 중 효과크기 최대 → 대표로 채택** |
| ✅ 채택 | TAX | ρ=-0.562 (중간) | 세율축 | **`RAD`보다 효과크기 큼 → 대표로 채택** |
| ✅ 채택 | PTRATIO | ρ=-0.556 (중간) | 없음 | 유의 + 중복 없음 |
| ✅ 채택 | ZN | ρ=+0.438 (중간) | 없음 | 유의 + 중복 없음 |
| ✅ 채택 | CHAS | 미제공 | 해당 없음 | 집단 차이 유의(인접1 고가) → 더미로 투입 |
| 🟡 후보 | NOX | ρ=-0.563 (중간) | 도심축 | 단독 관계는 유의하나 **`INDUS`와 중복 → 대체 가능** |
| 🟡 후보 | CRIM | ρ=-0.559 (중간) | 도심축 | 단독 관계는 유의하나 **강한 쌍 6개로 중복 최다** |
| 🟡 후보 | AGE | ρ=-0.548 (중간) | 도심축 | 단독 관계는 유의하나 **`INDUS`와 중복 → 대체 가능** |
| 🟡 후보 | DIS | ρ=+0.446 (중간) | 도심축 | 단독 관계는 유의하나 **`INDUS`에 대표를 양보** |
| 🟡 후보 | B | ρ=+0.186 (약함) | 없음 | 관계는 약하나 **가장 독립적** → 모델링 후 계수 유의성으로 판정 |
| ❌ 제외 | RAD | ρ=-0.347 (중간) | 세율축 | **중복 있는 변수 중 효과크기 최약 + `TAX`와 중복** → 남길 이유 없음 |
